# Day 2 · 한국어 회의 Agent

하나의 입력을 8개 차시 동안 확장합니다. 웹사이트 확인이 아니라 코드·명령·test·결과 파일을 직접 다루며, 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [1]:
from pathlib import Path
import importlib.util, json, subprocess, sys

def find_workspace(start):
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-day1.txt").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("WORKSPACE_ROOT_NOT_FOUND")

ROOT = find_workspace(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": ROOT.name, "python": sys.version.split()[0]})

{'workspace': 'llm-agent-and-workflow-automation', 'python': '3.12.12'}


In [2]:
# 최초 1회. 없는 핵심 library가 있을 때만 현재 Notebook Kernel에 설치합니다.
required = ["pydantic", "pytest", "langchain_core", "langgraph"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-day1.txt")],
        check=True,
    )
print({"missing_before_install": missing, "environment_ready": True})

# 실제 STT를 실행할 사람만 requirements-stt-optional.txt를 별도로 설치합니다.

{'missing_before_install': [], 'environment_ready': True}


In [3]:
OUT = ROOT / "output/course-labs/day2"
OUT.mkdir(parents=True, exist_ok=True)

def save_json(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print({"saved": str(path.relative_to(ROOT))})
    return path

def run_command(*args, cwd=ROOT):
    completed = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    display_args = list(args)
    if display_args and display_args[0] == sys.executable:
        display_args[0] = "python"
    result = {
        "command": " ".join(display_args),
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout.strip().splitlines()[-5:],
        "stderr_tail": completed.stderr.strip().splitlines()[-5:],
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

## 1차시 · 오디오 계약과 Metadata

모델을 실행하기 전에 파일 존재·길이·channel·sample rate를 확인합니다. 깨진 입력을 모델 오류로 오해하지 않는 첫 경계입니다.

In [4]:
import wave
from src.meeting_demo import ensure_workspace_path

audio_path = ensure_workspace_path(ROOT / "data/meeting_sample_ko_12min.wav", ROOT)
with wave.open(str(audio_path), "rb") as wav:
    audio_metadata = {
        "status": "SUCCESS",
        "seconds": round(wav.getnframes() / wav.getframerate(), 2),
        "sample_rate": wav.getframerate(),
        "channels": wav.getnchannels(),
        "sample_width": wav.getsampwidth(),
    }
save_json("01_audio_metadata.json", audio_metadata)
audio_metadata

{'saved': 'output/course-labs/day2/01_audio_metadata.json'}


{'status': 'SUCCESS',
 'seconds': 1017.46,
 'sample_rate': 22050,
 'channels': 1,
 'sample_width': 2}

## 2차시 · STT Adapter와 Timestamp Segment

`RUN_STT_LIVE=False`가 기본입니다. 모든 수강생은 reviewed fixture로 같은 segment 계약을 먼저 확인하고, 모델이 준비된 컴퓨터만 faster-whisper를 선택 실행합니다.

In [5]:
from src.meeting_demo import parse_transcript, run_demo

transcript_path = ROOT / "data/meeting_sample_ko_12min.txt"
transcript_text = transcript_path.read_text(encoding="utf-8")
RUN_STT_LIVE = False
if RUN_STT_LIVE:
    stt_run = run_demo(
        audio_path=audio_path,
        transcript_path=transcript_path,
        output_dir=OUT / "live-stt",
        model_size="small", device="cpu", compute_type="int8",
        language="ko", local_files_only=True,
    )
    stt_segments = stt_run["segments"]
    stt_status = {"provider_used": stt_run["mode"], "fallback_reason": stt_run["fallback_reason"]}
else:
    stt_segments = parse_transcript(transcript_text)
    stt_status = {"provider_used": "reviewed_fixture", "fallback_reason": "LIVE_STT_NOT_SELECTED"}
transcript_result = {**stt_status, "segment_count": len(stt_segments), "segments": stt_segments}
save_json("02_transcript.json", transcript_result)
{"provider_used": stt_status["provider_used"], "segment_count": len(stt_segments)}

{'saved': 'output/course-labs/day2/02_transcript.json'}


{'provider_used': 'reviewed_fixture', 'segment_count': 45}

## 3차시 · STT 품질 Gate

문자가 생성됐다는 사실과 업무에 사용 가능한 상태를 분리합니다. 정상 fixture는 READY, 빈 전사는 HOLD가 되어야 합니다.

In [6]:
from src.meeting_demo import build_quality_gate

ready_gate = build_quality_gate(
    stt_segments, mode="local_stt",
    metadata={"language": "ko", "language_probability": 0.99},
    reference_similarity=1.0,
)
hold_gate = build_quality_gate([], mode="fixture", metadata={"language": "ko"})
quality_result = {"normal": ready_gate, "boundary": hold_gate}
assert ready_gate["decision"] == "READY"
assert hold_gate["decision"] == "HOLD"
save_json("03_quality_gate.json", quality_result)
quality_result

{'saved': 'output/course-labs/day2/03_quality_gate.json'}


{'normal': {'decision': 'READY',
  'flagged_segment_count': 0,
  'reasons': [],
  'human_decision_required': True,
  'detected_language': 'ko',
  'language_probability': 0.99,
  'segment_count': 45,
  'text_length': 5141,
  'reference_similarity': 1.0,
  'minimum_reference_similarity': 0.8},
 'boundary': {'decision': 'HOLD',
  'flagged_segment_count': 0,
  'reasons': ['NO_SPEECH_SEGMENTS',
   'STT_FALLBACK_USED',
   'TRANSCRIPT_TOO_SHORT'],
  'human_decision_required': True,
  'detected_language': 'ko',
  'language_probability': None,
  'segment_count': 0,
  'text_length': 0,
  'reference_similarity': None,
  'minimum_reference_similarity': None}}

## 4차시 · MeetingBrief Schema

자연스러운 문장보다 필수 field·날짜 형식·evidence ID·사람 승인 정책을 먼저 고정합니다.

In [7]:
from pydantic import ValidationError
from src.langchain_lab import ActionItem, MeetingBrief, fixture_payload

meeting_brief = MeetingBrief.model_validate(fixture_payload())
try:
    ActionItem(task="근거 없는 할 일", owner="미정", due_date="2026-08-30", evidence_ids=[])
    schema_boundary = {"status": "UNEXPECTED_SUCCESS"}
except ValidationError as exc:
    schema_boundary = {"status": "EXPECTED_FAILURE", "error_code": exc.errors()[0]["type"]}
schema_result = {"normal": meeting_brief.model_dump(mode="json"), "boundary": schema_boundary}
save_json("04_meeting_schema.json", schema_result)
{"title": meeting_brief.title, "boundary": schema_boundary}

{'saved': 'output/course-labs/day2/04_meeting_schema.json'}


{'title': '고객 문의 자동화 PoC 범위 회의',
 'boundary': {'status': 'EXPECTED_FAILURE', 'error_code': 'too_short'}}

## 5차시 · Evidence-preserving Chunk

글자 수로 자르더라도 발화 ID와 overlap을 보존합니다. 너무 작은 chunk 설정은 명시적으로 실패시킵니다.

In [8]:
from src.course_services.meeting_service import chunk_transcript_segments

chunks = chunk_transcript_segments(stt_segments, max_chars=900, overlap_segments=1)
try:
    chunk_transcript_segments(stt_segments, max_chars=20)
    chunk_boundary = {"status": "UNEXPECTED_SUCCESS"}
except ValueError as exc:
    chunk_boundary = {"status": "EXPECTED_FAILURE", "error_code": str(exc)}
chunk_result = {"chunks": chunks, "boundary": chunk_boundary}
save_json("05_meeting_chunks.json", chunk_result)
{"chunk_count": len(chunks), "boundary": chunk_boundary}

{'saved': 'output/course-labs/day2/05_meeting_chunks.json'}


{'chunk_count': 7,
 'boundary': {'status': 'EXPECTED_FAILURE',
  'error_code': 'MAX_CHARS_TOO_SMALL'}}

## 6차시 · LangChain MeetingBrief Pipeline

fixture·Ollama·OpenAI가 같은 `MeetingBrief` 계약을 반환하도록 Prompt·Adapter·Parser·Policy를 분리합니다.

In [9]:
from src.langchain_lab import run_langchain_lab

RUN_OLLAMA_LIVE = False
provider = "ollama" if RUN_OLLAMA_LIVE else "fixture"
chain_result = run_langchain_lab(transcript_text, provider=provider, allow_fallback=True)
assert chain_result["checks"]["schema_valid"] is True
assert chain_result["result"]["automatic_email"] is False
save_json("06_meeting_brief.json", chain_result)
{key: chain_result[key] for key in ("provider_requested", "provider_used", "fallback_reason")}

{'saved': 'output/course-labs/day2/06_meeting_brief.json'}


{'provider_requested': 'fixture',
 'provider_used': 'fixture',
 'fallback_reason': None}

## 7차시 · Action Item 근거 검증

LLM이 만든 Action Item의 evidence ID가 실제 transcript에 없으면 HOLD합니다.

In [10]:
from src.course_services.meeting_service import validate_action_evidence

known_ids = {segment["id"] for segment in stt_segments}
normal_errors = validate_action_evidence(chain_result["result"]["action_items"], known_segment_ids=known_ids)
boundary_errors = validate_action_evidence(
    [{"task": "근거 없는 발행", "evidence_ids": ["s999"]}],
    known_segment_ids=known_ids,
)
evidence_result = {"normal_errors": normal_errors, "boundary_errors": boundary_errors}
assert normal_errors == []
assert boundary_errors == ["ACTION_1_UNKNOWN_EVIDENCE:s999"]
save_json("07_evidence_validation.json", evidence_result)
evidence_result

{'saved': 'output/course-labs/day2/07_evidence_validation.json'}


{'normal_errors': [], 'boundary_errors': ['ACTION_1_UNKNOWN_EVIDENCE:s999']}

## 8차시 · Golden Set과 Day 2 Scorecard

하루 결과를 현재 코드로 다시 생성하고 focused test를 실행합니다.

In [11]:
from src.course_services.course_demo import build_course_demo

day2_scorecard = build_course_demo(2, workspace_root=ROOT)
focused_test = run_command(sys.executable, "-m", "pytest", "-q", "tests/test_meeting_agent_workflow.py", "tests/test_course_services.py")
day2_scorecard["focused_test"] = focused_test
assert day2_scorecard["decision"] == "READY"
assert focused_test["returncode"] == 0
save_json("08_day2_scorecard.json", day2_scorecard)
{"decision": day2_scorecard["decision"], "metrics": day2_scorecard["metrics"]}

{
  "command": "python -m pytest -q tests/test_meeting_agent_workflow.py tests/test_course_services.py",
  "returncode": 0,
  "stdout_tail": [
    ".......................                                                  [100%]",
    "23 passed in 0.18s"
  ],
  "stderr_tail": []
}
{'saved': 'output/course-labs/day2/08_day2_scorecard.json'}


{'decision': 'READY',
 'metrics': {'segment_count': 45,
  'chunk_count': 7,
  'action_item_count': 5,
  'evidence_error_count': 0}}

## 완료 확인

- Day 2의 1~8차시 결과 파일을 확인했습니다.
- 정상 경로와 가장 중요한 실패 경로를 모두 실행했습니다.
- 외부 쓰기와 자동 메일이 기본값 `false`임을 확인했습니다.
- Codex·Claude Code 결과는 test와 diff를 사람이 검토한 뒤에만 반영합니다.